In [ ]:
import os
import json
from PIL import Image

# 경로 설정
json_path = "./053.대용량 손글씨 OCR 데이터/01.데이터/1.Training/라벨링데이터/라벨/HW-OCR/4.Validation/P.Paper/R.Free/IMG_OCR_53_4PR_09180.json"
img_path = "./053.대용량 손글씨 OCR 데이터/01.데이터/1.Training/원천데이터/TS6/HW-OCR/4.Validation/P.Paper/R.Free/IMG_OCR_53_4PR_09180.png"
save_dir = "./crops_09180"

# 저장 폴더 생성
os.makedirs(save_dir, exist_ok=True)

# 파일 이름에서 이미지 ID 추출
img_id = os.path.basename(img_path).split(".")[0]

# JSON 로드
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 이미지 로드
image = Image.open(img_path)

# 함수: bbox 좌표 변환
def get_bbox_coords(x_list, y_list):
    xmin, xmax = min(x_list), max(x_list)
    ymin, ymax = min(y_list), max(y_list)
    return xmin, ymin, xmax, ymax

# 함수: 파일명에 사용 불가능한 문자 제거
def sanitize_filename(name):
    return "".join(c for c in name if c.isalnum() or c in ('_', '-', '.', '(', ')')).rstrip()

# 단어별 bbox 잘라 저장
for box in data["bbox"]:
    word = box["data"]
    box_id = box["id"]
    x, y = box["x"], box["y"]
    xmin, ymin, xmax, ymax = get_bbox_coords(x, y)

    # 유효성 체크
    if xmax <= xmin or ymax <= ymin:
        print(f"[!] 잘못된 bbox 스킵: {word}")
        continue

    try:
        cropped = image.crop((xmin, ymin, xmax, ymax))
        safe_word = sanitize_filename(word)
        save_path = os.path.join(save_dir, f"{img_id}_{box_id:02d}_{safe_word}.png")
        cropped.save(save_path)
        print(f"✅ 저장됨: {save_path}")
    except Exception as e:
        print(f"[에러] {word}: {e}")

print("✅ 완료: 단일 이미지 전처리 끝!")


In [ ]:
import os
import json
from PIL import Image

# 경로 설정
json_dir = "./053.대용량 손글씨 OCR 데이터/01.데이터/1.Training/라벨링데이터/라벨/HW-OCR/4.Validation/P.Paper/R.Free"
image_root_dirs = [
    "./053.대용량 손글씨 OCR 데이터/01.데이터/1.Training/원천데이터/TS6/HW-OCR/4.Validation/P.Paper/R.Free",
    "./053.대용량 손글씨 OCR 데이터/01.데이터/1.Training/원천데이터/TS7/HW-OCR/4.Validation/P.Paper/R.Free",
    "./053.대용량 손글씨 OCR 데이터/01.데이터/1.Training/원천데이터/TS8/HW-OCR/4.Validation/P.Paper/R.Free"
]
save_root = "./all_crops"
os.makedirs(save_root, exist_ok=True)

def get_bbox_coords(x_list, y_list):
    return min(x_list), min(y_list), max(x_list), max(y_list)

def sanitize_filename(name):
    return "".join(c for c in name if c.isalnum() or c in ('_', '-', '.', '(', ')')).rstrip()

def find_image_path(img_name):
    for dir_path in image_root_dirs:
        full_path = os.path.join(dir_path, img_name)
        if os.path.exists(full_path):
            return full_path
    return None

# 전처리 시작
json_files = [f for f in os.listdir(json_dir) if f.endswith(".json")]

for json_file in json_files:
    json_path = os.path.join(json_dir, json_file)
    img_id = os.path.splitext(json_file)[0]
    img_filename = f"{img_id}.png"
    img_path = find_image_path(img_filename)

    if img_path is None:
        print(f"🚫 이미지 없음: {img_filename}")
        continue

    try:
        image = Image.open(img_path)
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        save_dir = os.path.join(save_root, img_id)
        os.makedirs(save_dir, exist_ok=True)

        for box in data["bbox"]:
            word = box["data"]
            box_id = box["id"]
            x, y = box["x"], box["y"]
            xmin, ymin, xmax, ymax = get_bbox_coords(x, y)

            if xmax <= xmin or ymax <= ymin:
                continue

            cropped = image.crop((xmin, ymin, xmax, ymax))
            safe_word = sanitize_filename(word)
            save_path = os.path.join(save_dir, f"{img_id}_{box_id:02d}_{safe_word}.png")
            cropped.save(save_path)

        print(f"✅ 처리 완료: {img_id}")

    except Exception as e:
        print(f"❌ 에러 발생 - {json_file}: {e}")

print("🎉 전체 전처리 완료!")


In [ ]:
import os
import pandas as pd
from itertools import combinations
from collections import defaultdict

# crop 이미지 경로
crop_root = "./all_crops"
csv_path = "siamese_pairs.csv"

pairs = []
word_dict = defaultdict(list)  # {word: [image_path1, image_path2, ...]}

# === 각 사용자 폴더 순회 ===
for user_folder in os.listdir(crop_root):
    user_path = os.path.join(crop_root, user_folder)
    if not os.path.isdir(user_path):
        continue

    user_images = sorted(os.listdir(user_path))
    user_fullpaths = [os.path.join(user_path, img) for img in user_images]

    # Positive pair: 같은 사용자 내 조합
    for img1, img2 in combinations(user_fullpaths, 2):
        pairs.append((img1, img2, 1))

    # 단어별로 저장 (negative pair 만들기 위해)
    for img_path in user_fullpaths:
        filename = os.path.basename(img_path)
        try:
            word = filename.split('_')[-1].split('.')[0]  # 단어 추출
            word_dict[word].append(img_path)
        except:
            continue

# Negative pair: 다른 사용자지만 같은 단어
for word, paths in word_dict.items():
    if len(paths) < 2:
        continue

    # 서로 다른 사용자 폴더에서만 추출
    for i in range(len(paths)):
        for j in range(i + 1, len(paths)):
            user1 = os.path.basename(os.path.dirname(paths[i]))
            user2 = os.path.basename(os.path.dirname(paths[j]))
            if user1 != user2:
                pairs.append((paths[i], paths[j], 0))

print(f"총 쌍 개수: {len(pairs)}")

# === CSV 저장 ===
df = pd.DataFrame(pairs, columns=["img1", "img2", "label"])
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"📄 CSV 저장 완료: {csv_path}")


In [ ]:
import pandas as pd

df = pd.read_csv("siamese_pairs.csv")
print(df.head(10))


In [ ]:
import pandas as pd

df = pd.read_csv("./siamese_pairs.csv")
df.columns  # 컬럼명만 보기


In [ ]:
### 1. Dataset 클래스 정의 - csv 기반 이미지 쌍 로딩
## 위에서 만든 csv파일 기반으로 이미지 쌍과 라벨(0=다른 사람, 1=같은 사람)을 불러오는 SiameseDataset클래스 만들기

import torch
from torch.utils.data import Dataset
from PIL import Image
import pandas as pd
import torchvision.transforms as transforms

class SiameseDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.data = pd.read_csv(csv_path)
        self.transform = transform if transform else transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __getitem__(self, index):
        row = self.data.iloc[index]
        img1_path = row['img1']
        img2_path = row['img2']
        label = torch.tensor(row['label'], dtype=torch.float32)

        img1 = Image.open(img1_path).convert("RGB")
        img2 = Image.open(img2_path).convert("RGB")

        img1 = self.transform(img1)
        img2 = self.transform(img2)

        return img1, img2, label

    def __len__(self):
        return len(self.data)


In [ ]:
### 2. Siamese Network 모델 구조 정의(ResNet18 기반)

import torch
import torch.nn as nn
import torchvision.models as models

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        # 사전 학습된 ResNet18을 가져오고 마지막 FC층 제거
        resnet = models.resnet18(pretrained=True)
        modules = list(resnet.children())[:-1]  # 마지막 FC 레이어 제거
        self.feature_extractor = nn.Sequential(*modules)

        # 임베딩 벡터 차원: 512
        self.embedding_dim = 512

        # 두 벡터 간 차이로부터 최종 예측값 출력하는 분류기
        self.classifier = nn.Sequential(
            nn.Linear(self.embedding_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()  # BCE Loss를 쓰므로 Sigmoid
        )

    def forward_once(self, x):
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)  # Flatten
        return x

    def forward(self, img1, img2):
        emb1 = self.forward_once(img1)
        emb2 = self.forward_once(img2)
        diff = torch.abs(emb1 - emb2)  # 절댓값 차이 (L1 distance)
        out = self.classifier(diff)
        return out


In [ ]:
### 3. 손실 함수 정의(BCELose)

import torch.nn as nn

criterion = nn.BCELoss()


In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm


# 하이퍼파라미터
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 10
batch_size = 32
lr = 1e-4

# 데이터 전처리
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Dataset 로딩
csv_path = 'siamese_pairs.csv'
train_dataset = SiameseDataset(csv_path=csv_path, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# ✅ 모델, 손실 함수, 옵티마이저
model = SiameseNetwork().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# ✅ 학습 루프
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for img1, img2, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        img1, img2, label = img1.to(device), img2.to(device), label.to(device).float()

        optimizer.zero_grad()
        outputs = model(img1, img2).view(-1)  # batch 단위로 예측 결과 flatten
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        preds = (outputs > 0.5).float()
        correct += (preds == label).sum().item()
        total += label.size(0)

    acc = correct / total if total else 0
    print(f"✅ Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {acc:.4f}")


In [ ]:
import pandas as pd

# 원본 CSV 경로
df = pd.read_csv("siamese_pairs.csv")

# 샘플 수 설정 (예: 총 10000쌍 = positive 5000 + negative 5000)
n_samples = 10000

# Positive / Negative 비율 확인하고 균형 있게 샘플링
sampled_df = df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(n=n_samples // 2, random_state=42)
)

# 저장
sampled_df.to_csv("siamese_pairs_sampled.csv", index=False)
print("✅ 샘플링 완료: siamese_pairs_sampled.csv 저장됨")


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

# 하이퍼파라미터 및 device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 10
batch_size = 32
lr = 1e-4

# 전처리 정의
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 데이터셋 로딩
train_dataset = SiameseDataset(csv_path='siamese_pairs_sampled.csv', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 모델, 손실 함수, 옵티마이저
model = SiameseNetwork().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# 학습 루프
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for img1, img2, label in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        img1, img2, label = img1.to(device), img2.to(device), label.to(device).float()

        optimizer.zero_grad()
        output = model(img1, img2).squeeze()
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = (output > 0.5).float()
        correct += (preds == label).sum().item()
        total += label.size(0)

    acc = correct / total
    print(f"✅ Epoch {epoch+1}/{epochs} | Loss: {epoch_loss:.4f} | Accuracy: {acc:.4f}")


In [ ]:
### validation 이미지 전처리

import os
import json
from PIL import Image

# ✅ Validation 데이터의 라벨링 및 이미지 경로
val_json_root = "./053.대용량 손글씨 OCR 데이터/01.데이터/2.Validation/라벨링데이터/라벨/HW-OCR/4.Validation"
val_img_root = "./053.대용량 손글씨 OCR 데이터/01.데이터/2.Validation/원천데이터/VS/HW-OCR/4.Validation"

# ✅ 하위 디렉토리들: P.Paper/R.Free, T.Tablet/R.Free
subfolders = [
    ("P.Paper/R.Free", "val_crops_paper"),
    ("T.Tablet/R.Free", "val_crops_tablet")
]

def get_bbox_coords(x_list, y_list):
    return min(x_list), min(y_list), max(x_list), max(y_list)

def sanitize_filename(name):
    return "".join(c for c in name if c.isalnum() or c in ('_', '-', '.', '(', ')')).rstrip()

# ✅ 각 subfolder 별로 반복 처리
for subfolder, save_root in subfolders:
    json_dir = os.path.join(val_json_root, subfolder)
    image_dir = os.path.join(val_img_root, subfolder)
    os.makedirs(save_root, exist_ok=True)

    json_files = [f for f in os.listdir(json_dir) if f.endswith(".json")]

    for json_file in json_files:
        json_path = os.path.join(json_dir, json_file)
        img_id = os.path.splitext(json_file)[0]
        img_filename = f"{img_id}.png"
        img_path = os.path.join(image_dir, img_filename)

        if not os.path.exists(img_path):
            print(f"🚫 이미지 없음: {img_filename}")
            continue

        try:
            image = Image.open(img_path)
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            save_dir = os.path.join(save_root, img_id)
            os.makedirs(save_dir, exist_ok=True)

            for box in data["bbox"]:
                word = box["data"]
                box_id = box["id"]
                x, y = box["x"], box["y"]
                xmin, ymin, xmax, ymax = get_bbox_coords(x, y)

                if xmax <= xmin or ymax <= ymin:
                    continue

                cropped = image.crop((xmin, ymin, xmax, ymax))
                safe_word = sanitize_filename(word)
                save_path = os.path.join(save_dir, f"{img_id}_{box_id:02d}_{safe_word}.png")
                cropped.save(save_path)

            print(f"✅ 처리 완료: {subfolder} - {img_id}")

        except Exception as e:
            print(f"❌ 에러 발생 - {subfolder} - {json_file}: {e}")

print("🎉 Validation (P.Paper + T.Tablet) 전처리 완료!")

